# EXPL_99 — Explainability Phase Summary

Run this notebook after EXPL_01–EXPL_06 have completed. It inventories the
generated evidence and produces the final Markdown handoff document.


In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "explainability.yaml"

from src.ontario_peak_risk.explainability.common import (
    load_explainability_config,
    ensure_phase_directories,
    load_final_metadata,
)
config, project_root = load_explainability_config(CONFIG_PATH)
phase_paths = ensure_phase_directories(config, project_root)
metadata = load_final_metadata(config, project_root)

print("Project root:", project_root)
print("RF:", metadata["rf"]["algorithm"], "| horizons:", metadata["rf"]["horizons"])
print("XGB:", metadata["xgb"]["algorithm"], "| horizons:", metadata["xgb"]["horizons"])
print("Explainability output:", phase_paths["outputs_dir"])


Project root: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk
RF: RandomForestRegressor | horizons: 24
XGB: XGBoostClassifier | horizons: 24
Explainability output: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\outputs\explainability


In [2]:
from src.ontario_peak_risk.explainability.reporting import write_phase_overview

expected = {
    "EXPL_01": phase_paths["docs_dir"] / "EXPL_01_Global_Feature_Importance.md",
    "EXPL_02": phase_paths["docs_dir"] / "EXPL_02_Global_SHAP.md",
    "EXPL_03": phase_paths["docs_dir"] / "EXPL_03_Weather_Influence.md",
    "EXPL_06": phase_paths["docs_dir"] / "EXPL_06_Operational_Local_Explanation.md",
}

status = pd.DataFrame([
    {
        "section": name,
        "path": str(path),
        "status": "PASS" if path.exists() else "MISSING",
    }
    for name, path in expected.items()
])

display(status)

if status["status"].ne("PASS").any():
    raise ValueError(
        "Explainability documentation is incomplete. "
        "Run the missing notebooks before EXPL_99."
    )


,section,path,status
0,EXPL_01,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,PASS
1,EXPL_02,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,PASS
2,EXPL_03,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,PASS
3,EXPL_06,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,PASS


In [3]:
importance = pd.read_csv(
    phase_paths["outputs_dir"] / "EXPL_01_global_importance_summary.csv"
)
shap_global = pd.read_csv(
    phase_paths["outputs_dir"] / "EXPL_02_shap_global_by_horizon.csv"
)
weather = pd.read_csv(
    phase_paths["outputs_dir"] / "EXPL_03_weather_shap_summary.csv"
)

findings = []

for task in ["rf", "xgb"]:
    top_imp = (
        importance.loc[importance["task"].eq(task)]
        .nlargest(3, "mean_importance_pct")
    )
    names = ", ".join(top_imp["feature"].astype(str))
    findings.append(
        f"{task.upper()} top built-in features: {names}."
    )

    top_shap = (
        shap_global.loc[shap_global["task"].eq(task)]
        .groupby("feature", observed=True)["mean_abs_shap"]
        .mean()
        .nlargest(3)
        .index
    )
    findings.append(
        f"{task.upper()} top SHAP features: {', '.join(map(str, top_shap))}."
    )

for _, row in weather.groupby("task", observed=True)["weather_share_pct"].mean().reset_index().iterrows():
    findings.append(
        f"{row['task'].upper()} mean weather share of SHAP importance "
        f"across representative horizons: {row['weather_share_pct']:.2f}%."
    )

write_phase_overview(
    phase_paths["docs_dir"] / "Explainability_Phase_Overview.md",
    completed_sections=[
        "EXPL_00 Design and Scope",
        "EXPL_01 Global Feature Importance",
        "EXPL_02 Global SHAP",
        "EXPL_03 Weather Influence",
        "EXPL_04 Local Historical Explanations",
        "EXPL_05 Horizon Comparison",
        "EXPL_06 Operational Local Explanation",
    ],
    key_findings=findings,
)

print("\n".join(findings))
print("\nEXPLAINABILITY PHASE RESULT: COMPLETE")


RF top built-in features: target_lag_24h, target_lag_48h, target_lag_168h.
RF top SHAP features: target_lag_24h, target_lag_48h, target_lag_168h.
XGB top built-in features: fsa, target_hour, target_season.
XGB top SHAP features: target_hour, origin__Temp (°C), target_lag_24h.
RF mean weather share of SHAP importance across representative horizons: 5.45%.
XGB mean weather share of SHAP importance across representative horizons: 17.36%.

EXPLAINABILITY PHASE RESULT: COMPLETE
